# EvoVariant-TR autonomous adaptation

Research-only execution. The 946-row locked cohort is never loaded here.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/content/EvoVariant')
DRIVE_ROOT = Path('/content/drive/MyDrive/EvoVariantTR')
print('root', ROOT)
print('drive', DRIVE_ROOT)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for name in ('reference','datasets','checkpoints','model_cache','runs','hpo','logs','state','exports'):
    (DRIVE_ROOT / name).mkdir(exist_ok=True)


In [ ]:
import subprocess
if not ROOT.exists():
    subprocess.run(['git','clone','-b','research/posthoc-foundation-adaptation','https://github.com/UtkarsHMer05/EvoVariant-TR-.git',str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','research/posthoc-foundation-adaptation'], check=True)
print(subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'], text=True).strip())
print(subprocess.check_output(['git','-C',str(ROOT),'status','--short','--branch'], text=True).strip())


In [ ]:
import shutil
ENV = Path('/content/caduceus-env')
UV = shutil.which('uv') or '/usr/local/bin/uv'
if not ENV.exists():
    subprocess.run([UV,'venv','--python','3.11',str(ENV)], check=True)
PY = str(ENV / 'bin' / 'python')
print(subprocess.check_output([PY,'-c','import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))'], text=True))


In [ ]:
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
subprocess.run([PY,'scripts/adaptation/hardware_probe.py'], check=True)
subprocess.run([PY,'scripts/adaptation/verify_data.py'], check=True)


In [ ]:
REFERENCE = DRIVE_ROOT / 'reference' / 'Homo_sapiens_assembly38.fasta'
if not REFERENCE.exists():
    archive = REFERENCE.with_name('hg38.fa.gz')
    subprocess.run(['wget','-c','https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz','-O',str(archive)], check=True)
    subprocess.run(['gunzip','-f',str(archive)], check=True)
    (REFERENCE.with_name('hg38.fa')).replace(REFERENCE)
subprocess.run([PY,'-c',f'from pyfaidx import Fasta; Fasta({str(REFERENCE)!r})'], check=True)
print(REFERENCE)


In [ ]:
SMOKE = ROOT / 'artifacts/adaptation/caduceus_smoke.json'
subprocess.run([PY,'scripts/adaptation/smoke_caduceus.py','--device','cuda','--cache-dir',str(DRIVE_ROOT/'model_cache'),'--output',str(SMOKE)], check=True)
print(SMOKE)


In [ ]:
HPO = DRIVE_ROOT / 'hpo' / 'caduceus_hpo.json'
subprocess.run([PY,'scripts/adaptation/run_caduceus_hpo.py','--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--output',str(HPO),'--device','cuda','--trials','8'], check=True)


In [ ]:
RUN = DRIVE_ROOT / 'runs' / 'caduceus_final'
subprocess.run([PY,'scripts/adaptation/train_caduceus.py','--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--output-dir',str(RUN),'--device','cuda','--stage','frozen','--resume',str(RUN/'latest.pt')], check=True)


In [ ]:
# After TRAIN-only selection closes, evaluate the 801-row validation cohort once.
HOLDOUT = DRIVE_ROOT / 'exports' / 'caduceus_validation.csv'
subprocess.run([PY,'scripts/adaptation/evaluate_caduceus.py','--reference',str(REFERENCE),'--cache-dir',str(DRIVE_ROOT/'model_cache'),'--checkpoint',str(RUN/'latest.pt'),'--output',str(HOLDOUT),'--split','validation','--device','cuda'], check=True)
